## E-Commerce Funnel Analysis — Exploratory Data Analysis (EDA)




In [ ]:
# Importing Libraries

import pandas as pd          
import matplotlib.pyplot as plt  
import seaborn as sns        
import warnings             
warnings.filterwarnings('ignore')
import pyodbc


sns.set_theme(style='darkgrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 5)  
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("All libraries loaded successfully!")


All libraries loaded successfully!


## Step 1 :  Database Connection 

In [6]:
# Database Connection and Interaction 


conn = pyodbc.connect(
    "DRIVER={SQL Server};"
    "SERVER=DESKTOP-EUH7TII;"
    "DATABASE=Ecommerce_Funnel;"
    "Trusted_Connection=yes;"
)

orders    = pd.read_sql("SELECT * FROM orders", conn)
customers = pd.read_sql("SELECT * FROM customers", conn)
items     = pd.read_sql("SELECT * FROM order_items", conn)
products  = pd.read_sql("SELECT * FROM products", conn)
payments  = pd.read_sql("SELECT * FROM payments", conn)
reviews   = pd.read_sql("SELECT * FROM reviews", conn)


print(" == All files loaded == ")
print()
print("Row counts:")
print(f"  orders    : {len(orders):,} rows")
print(f"  customers : {len(customers):,} rows")
print(f"  items     : {len(items):,} rows")
print(f"  products  : {len(products):,} rows")
print(f"  payments  : {len(payments):,} rows")
print(f"  reviews   : {len(reviews):,} rows")



 == All files loaded == 

Row counts:
  orders    : 99,441 rows
  customers : 99,441 rows
  items     : 112,650 rows
  products  : 32,951 rows
  payments  : 103,886 rows
  reviews   : 99,224 rows


In [ ]:
sns.countplot(orders['order_id'])
plt.show()


## Step 2 : Data Understanding


In [ ]:

print("=== ORDERS TABLE ===")
orders.head()


In [ ]:
# Check all column names and their data types
print("=== ORDERS — Column Info ===")
orders.info()


In [ ]:
# Check for missing values (nulls) in the orders table
print("=== ORDERS — Missing Values Count ===")
missing = orders.isnull().sum()
missing_pct = (orders.isnull().sum() / len(orders) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0])


In [ ]:
# Quick summary of all 6 tables 
print("Dataset Shapes:")
print(f"  orders    : {orders.shape}")
print(f"  customers : {customers.shape}")
print(f"  items     : {items.shape}")
print(f"  products  : {products.shape}")
print(f"  payments  : {payments.shape}")
print(f"  reviews   : {reviews.shape}")



## Step 3 : Data Cleaning 



In [ ]:
# Fix Datatype format 


date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col])

print(" == Date columns converted == ")
print()
print("Data types after conversion:")
print(orders[date_cols].dtypes)


## Step 4 : Normailized Data


In [ ]:
# Merge all tables into one big master DataFrame


# Step 1: orders + customers
df = orders.merge(customers, on='customer_id', how='left')

# Step 2: + items (sum price per order to get order total)
items_agg = items.groupby('order_id').agg(
    total_items  = ('order_item_id', 'count'),
    total_price  = ('price', 'sum'),
    total_freight= ('freight_value', 'sum')
).reset_index()

df = df.merge(items_agg, on='order_id', how='left')

# Step 3: + payments (total payment per order)
pay_agg = payments.groupby('order_id').agg(
    payment_type  = ('payment_type', 'first'),   
    payment_value = ('payment_value', 'sum'),     
    installments  = ('payment_installments', 'max')
).reset_index()

df = df.merge(pay_agg, on='order_id', how='left')

# Step 4: + reviews (first review per order)
rev_agg = reviews.groupby('order_id').agg(
    review_score = ('review_score', 'first')
).reset_index()

df = df.merge(rev_agg, on='order_id', how='left')

print(f"Master DataFrame created! Shape: {df.shape}")
print(f"   Rows: {len(df):,} | Columns: {df.shape[1]}")


In [ ]:
# Create useful calculated columns

# How many days from order placement to delivery?
df['days_to_deliver'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.days

# Was delivery late? (1 = yes, 0 = no)
df['is_late'] = (
    df['order_delivered_customer_date'] > df['order_estimated_delivery_date']
).astype(int)

# Extract year and month for trend analysis
df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M')
df['order_year']  = df['order_purchase_timestamp'].dt.year

print(" Calculated columns added:")
print("   - days_to_deliver")
print("   - is_late")
print("   - order_month")
print("   - order_year")



## Step 5 — Order Status Distribution 
Usecase : How orders are spread across different status?  



In [ ]:
# Count how many orders are in each status
status_counts = df['order_status'].value_counts().reset_index()
status_counts.columns = ['status', 'count']
status_counts['percentage'] = (status_counts['count'] / len(df) * 100).round(2)

print(status_counts.to_string(index=False))


NameError: name 'df' is not defined

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Chart 1: Bar chart ---
colors = sns.color_palette('Set2', len(status_counts))
axes[0].bar(status_counts['status'], status_counts['count'], color=colors, edgecolor='white')
axes[0].set_title('Order Count by Status', fontweight='bold')
axes[0].set_xlabel('Order Status')
axes[0].set_ylabel('Number of Orders')
axes[0].tick_params(axis='x', rotation=30)

# Add count labels on top of each bar
for i, row in status_counts.iterrows():
    axes[0].text(i, row['count'] + 200, f"{row['count']:,}",
                 ha='center', va='bottom', fontsize=9)

# --- Chart 2: Pie chart (top 5 only for clarity) ---
top5 = status_counts.head(5)
axes[1].pie(top5['count'], labels=top5['status'], autopct='%1.1f%%',
            colors=colors[:5], startangle=140, pctdistance=0.8)
axes[1].set_title('Order Status Share (Top 5)', fontweight='bold')

plt.suptitle('Order Status Overview', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('order_status.png', dpi=150, bbox_inches='tight')
plt.show()




## Step 6 — Funnel Drop-off Analysis 

Usecase : How many orders made it through each stage of the funnel?
          The goal is to find the biggest drop-off point.

**Funnel stages:**
- Stage 1: Order Placed (all orders)
- Stage 2: Payment Approved
- Stage 3: Handed to Shipping Carrier
- Stage 4: Delivered to Customer
- Stage 5: Customer Left a Review


In [ ]:
# Count orders at each funnel stage
# A NULL (missing) timestamp means that stage was never reached

stage1 = len(df)   # All orders placed

stage2 = df['order_approved_at'].notna().sum()          # approved
stage3 = df['order_delivered_carrier_date'].notna().sum() if 'order_delivered_carrier_date' in df.columns else df['order_delivered_carrier_date'].notna().sum()
stage4 = df['order_delivered_customer_date'].notna().sum()
stage5 = df['review_score'].notna().sum()

# Handle column name variation
carrier_col = 'order_delivered_carrier_date' if 'order_delivered_carrier_date' in df.columns else 'order_delivered_carrier_date'
stage3 = df[carrier_col].notna().sum() if carrier_col in df.columns else df.filter(like='carrier').iloc[:,0].notna().sum()

stages = {
    'Stage 1\nPlaced':    stage1,
    'Stage 2\nApproved':  stage2,
    'Stage 3\nShipped':   stage3,
    'Stage 4\nDelivered': stage4,
    'Stage 5\nReviewed':  stage5
}

# Build a summary table
funnel_df = pd.DataFrame({
    'Stage': list(stages.keys()),
    'Orders': list(stages.values())
})
funnel_df['% of Total']  = (funnel_df['Orders'] / stage1 * 100).round(2)
funnel_df['Drop from Prev'] = funnel_df['Orders'].diff().fillna(0).abs().astype(int)
funnel_df['Drop %'] = (funnel_df['Drop from Prev'] / funnel_df['Orders'].shift(1) * 100).round(2).fillna(0)

print(funnel_df.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

stage_labels = ['Placed', 'Approved', 'Shipped', 'Delivered', 'Reviewed']
stage_values = list(stages.values())
bar_colors   = ['#2196F3','#4CAF50','#FF9800','#9C27B0','#F44336']

# Chart 1: Funnel Bar Chart 
bars = axes[0].barh(stage_labels[::-1], stage_values[::-1],
                     color=bar_colors[::-1], edgecolor='white', height=0.6)
axes[0].set_title('Funnel Drop-off — Order Count at Each Stage', fontweight='bold')
axes[0].set_xlabel('Number of Orders')

# Add count labels inside bars
for bar, val in zip(bars, stage_values[::-1]):
    axes[0].text(val * 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', ha='center',
                 color='white', fontweight='bold', fontsize=11)

# Chart 2: Conversion Rate 
conv_rates = [v / stage1 * 100 for v in stage_values]
axes[1].bar(stage_labels, conv_rates, color=bar_colors, edgecolor='white')
axes[1].set_title('Conversion Rate at Each Stage (%)', fontweight='bold')
axes[1].set_xlabel('Funnel Stage')
axes[1].set_ylabel('% of Total Orders')
axes[1].set_ylim(0, 110)

for i, (v, c) in enumerate(zip(stage_values, conv_rates)):
    axes[1].text(i, c + 1.5, f'{c:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('E-Commerce Funnel Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('funnel_analysis.png', dpi=150, bbox_inches='tight')
plt.show()




## Step 7 — Revenue Analysis 

Usecase : How much revenue did the platform make?  
          How is order value distributed — are most orders cheap or expensive?


In [ ]:
# Filter only delivered orders for revenue analysis
delivered = df[df['order_status'] == 'delivered'].copy()

print("=== Revenue Summary (Delivered Orders Only) ===")
print(f"  Total Orders Delivered : {len(delivered):,}")
print(f"  Total Revenue (BRL)    : R$ {delivered['total_price'].sum():,.2f}")
print(f"  Average Order Value    : R$ {delivered['total_price'].mean():,.2f}")
print(f"  Median Order Value     : R$ {delivered['total_price'].median():,.2f}")
print(f"  Min Order Value        : R$ {delivered['total_price'].min():,.2f}")
print(f"  Max Order Value        : R$ {delivered['total_price'].max():,.2f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Chart 1: Distribution of order values 
# Remove extreme outliers (top 1%) so the chart is readable
price_cap = delivered['total_price'].quantile(0.99)
clean_price = delivered[delivered['total_price'] <= price_cap]['total_price']

axes[0].hist(clean_price, bins=50, color='#2196F3', edgecolor='white', alpha=0.85)
axes[0].axvline(clean_price.mean(),   color='red',    linestyle='--', linewidth=2, label=f'Mean: R${clean_price.mean():.0f}')
axes[0].axvline(clean_price.median(), color='orange', linestyle='--', linewidth=2, label=f'Median: R${clean_price.median():.0f}')
axes[0].set_title('Distribution of Order Values (excluding top 1%)', fontweight='bold')
axes[0].set_xlabel('Order Value (BRL R$)')
axes[0].set_ylabel('Number of Orders')
axes[0].legend()

# Chart 2: Revenue by order status 
rev_by_status = df.groupby('order_status')['total_price'].sum().sort_values(ascending=False)
axes[1].bar(rev_by_status.index, rev_by_status.values,
            color=sns.color_palette('Set2', len(rev_by_status)), edgecolor='white')
axes[1].set_title('Total Revenue by Order Status', fontweight='bold')
axes[1].set_xlabel('Status')
axes[1].set_ylabel('Total Revenue (BRL)')
axes[1].tick_params(axis='x', rotation=30)

for i, v in enumerate(rev_by_status.values):
    axes[1].text(i, v + 5000, f'R${v/1e6:.1f}M', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Revenue Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('revenue_analysis.png', dpi=150, bbox_inches='tight')
plt.show()




## Step 8 — Delivery Time Analysis 

Usecase : How fast does delivery happen?  
          Are there orders that take unusually long?  
          Slow delivery is often linked to customer dissatisfaction.


In [ ]:
# Filter only delivered orders with valid delivery times
deliver_df = df[(df['order_status'] == 'delivered') &
                (df['days_to_deliver'].notna()) &
                (df['days_to_deliver'] > 0)].copy()

print("=== Delivery Time Stats ===")
print(f"  Average days to deliver : {deliver_df['days_to_deliver'].mean():.1f} days")
print(f"  Median days to deliver  : {deliver_df['days_to_deliver'].median():.1f} days")
print(f"  Fastest delivery        : {deliver_df['days_to_deliver'].min():.0f} days")
print(f"  Slowest delivery        : {deliver_df['days_to_deliver'].max():.0f} days")
print()

# Delivery time buckets
bins   = [0, 3, 7, 14, 21, 30, 9999]
labels = ['0-3 days', '4-7 days', '8-14 days', '15-21 days', '22-30 days', '30+ days']
deliver_df['delivery_bucket'] = pd.cut(deliver_df['days_to_deliver'], bins=bins, labels=labels)

bucket_counts = deliver_df['delivery_bucket'].value_counts().sort_index()
bucket_pct    = (bucket_counts / len(deliver_df) * 100).round(1)

print("Delivery Time Breakdown:")
for b, c, p in zip(bucket_counts.index, bucket_counts.values, bucket_pct.values):
    print(f"  {b:15s}: {c:6,} orders ({p}%)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Chart 1: Delivery time distribution 
cap = deliver_df['days_to_deliver'].quantile(0.98)
clean_days = deliver_df[deliver_df['days_to_deliver'] <= cap]['days_to_deliver']

axes[0].hist(clean_days, bins=40, color='#4CAF50', edgecolor='white', alpha=0.85)
axes[0].axvline(clean_days.mean(),   color='red',    linestyle='--', linewidth=2,
                label=f'Mean: {clean_days.mean():.1f} days')
axes[0].axvline(clean_days.median(), color='orange', linestyle='--', linewidth=2,
                label=f'Median: {clean_days.median():.1f} days')
axes[0].set_title('Distribution of Delivery Times', fontweight='bold')
axes[0].set_xlabel('Days to Deliver')
axes[0].set_ylabel('Number of Orders')
axes[0].legend()

# Chart 2: Bucket bar chart 
bucket_colors = ['#1a9641','#a6d96a','#fdae61','#f46d43','#d73027','#a50026']
axes[1].bar(bucket_counts.index, bucket_counts.values,
            color=bucket_colors[:len(bucket_counts)], edgecolor='white')
axes[1].set_title('Orders by Delivery Time Bucket', fontweight='bold')
axes[1].set_xlabel('Delivery Time Range')
axes[1].set_ylabel('Number of Orders')
axes[1].tick_params(axis='x', rotation=25)

for i, (c, p) in enumerate(zip(bucket_counts.values, bucket_pct.values)):
    axes[1].text(i, c + 100, f'{p}%', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Delivery Speed Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('delivery_time.png', dpi=150, bbox_inches='tight')
plt.show()




## Step 9 - Payment Method Analysis 

Usecase : Which payment methods do customers prefer?  
          Does payment method affect whether the order gets delivered?


In [ ]:
# Payment type distribution
pay_dist = df['payment_type'].value_counts()
pay_pct  = (pay_dist / len(df) * 100).round(2)

print("=== Payment Method Distribution ===")
for p, c, pct in zip(pay_dist.index, pay_dist.values, pay_pct.values):
    print(f"  {p:15s}: {c:6,} orders ({pct}%)")

print()

# Delivery rate per payment type
pay_delivery = df.groupby('payment_type').apply(
    lambda x: (x['order_status'] == 'delivered').mean() * 100
).round(2).sort_values(ascending=False)

print("=== Delivery Rate by Payment Type ===")
for p, r in pay_delivery.items():
    print(f"  {p:15s}: {r}% delivered")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
pal = sns.color_palette('Set2', len(pay_dist))

# Chart 1: Payment type volume 
axes[0].bar(pay_dist.index, pay_dist.values, color=pal, edgecolor='white')
axes[0].set_title('Orders by Payment Type', fontweight='bold')
axes[0].set_xlabel('Payment Type')
axes[0].set_ylabel('Number of Orders')
for i, v in enumerate(pay_dist.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=9)

# Chart 2: Pie chart 
axes[1].pie(pay_dist.values, labels=pay_dist.index, autopct='%1.1f%%',
            colors=pal, startangle=140, pctdistance=0.8)
axes[1].set_title('Payment Type Share', fontweight='bold')

# Chart 3: Delivery rate per payment type 
axes[2].bar(pay_delivery.index, pay_delivery.values, color=pal, edgecolor='white')
axes[2].set_title('Delivery Rate (%) by Payment Type', fontweight='bold')
axes[2].set_xlabel('Payment Type')
axes[2].set_ylabel('% Delivered')
axes[2].set_ylim(0, 110)
for i, v in enumerate(pay_delivery.values):
    axes[2].text(i, v + 1, f'{v}%', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Payment Method Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('payment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()




## Step 10 - Review Score Analysis 

Usecase : How satisfied are customers after receiving their orders?  
          A score of 1 = very unhappy, 5 = very happy.


In [ ]:
# Review score distribution
rev_dist = df['review_score'].value_counts().sort_index()
rev_pct  = (rev_dist / rev_dist.sum() * 100).round(2)

print("=== Review Score Distribution ===")
for score, count, pct in zip(rev_dist.index, rev_dist.values, rev_pct.values):
    bar = '█' * int(pct / 2)
    print(f"  Score {int(score)}: {count:6,} ({pct:5.1f}%) {bar}")

print()
print(f"  Average Score: {df['review_score'].mean():.2f} / 5.00")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

score_colors = {1:'#d73027', 2:'#f46d43', 3:'#fdae61', 4:'#a6d96a', 5:'#1a9641'}
colors_list  = [score_colors[s] for s in rev_dist.index.astype(int)]

# Chart 1: Bar chart of scores 
axes[0].bar(rev_dist.index.astype(int), rev_dist.values, color=colors_list, edgecolor='white', width=0.6)
axes[0].set_title('Review Score Distribution', fontweight='bold')
axes[0].set_xlabel('Review Score (1=Very Bad, 5=Very Good)')
axes[0].set_ylabel('Number of Reviews')
axes[0].set_xticks([1,2,3,4,5])

for i, (s, v, p) in enumerate(zip(rev_dist.index, rev_dist.values, rev_pct.values)):
    axes[0].text(s, v + 200, f'{p}%', ha='center', fontsize=10, fontweight='bold')

# Chart 2: Avg review score vs delivery speed 
score_vs_days = df.groupby('review_score')['days_to_deliver'].mean().dropna()

bar2 = axes[1].bar(score_vs_days.index.astype(int), score_vs_days.values,
                    color=colors_list, edgecolor='white', width=0.6)
axes[1].set_title('Avg Delivery Days vs Review Score\n(Slower delivery → worse reviews?)',
                   fontweight='bold')
axes[1].set_xlabel('Review Score')
axes[1].set_ylabel('Avg Days to Deliver')
axes[1].set_xticks([1,2,3,4,5])

for s, v in score_vs_days.items():
    axes[1].text(s, v + 0.2, f'{v:.1f}d', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Customer Review Score Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('review_scores.png', dpi=150, bbox_inches='tight')
plt.show()




## Step 11 — Late Delivery Impact 

Usecase : We know slow delivery is bad — but **how bad**?  
          This step compares on-time vs late deliveries  
          to see the real impact on review scores.


In [ ]:
# Filter only delivered orders with valid late flag
late_df = df[(df['order_status'] == 'delivered') &
             (df['is_late'].notna()) &
             (df['review_score'].notna())].copy()

late_df['delivery_type'] = late_df['is_late'].map({1: 'Late', 0: 'On Time'})

# Summary stats
summary = late_df.groupby('delivery_type').agg(
    order_count       = ('order_id', 'count'),
    avg_review_score  = ('review_score', 'mean'),
    pct_bad_reviews   = ('review_score', lambda x: (x <= 2).mean() * 100),
    avg_days_to_deliver = ('days_to_deliver', 'mean')
).round(2)

print("=== Late vs On-Time Delivery Comparison ===")
print(summary.to_string())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Chart 1: Count of late vs on-time 
late_counts = late_df['delivery_type'].value_counts()
axes[0].bar(late_counts.index, late_counts.values,
            color=['#d73027','#1a9641'], edgecolor='white', width=0.4)
axes[0].set_title('Late vs On-Time Deliveries', fontweight='bold')
axes[0].set_ylabel('Number of Orders')
for i, v in enumerate(late_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

# Chart 2: Avg review score 
avg_scores = late_df.groupby('delivery_type')['review_score'].mean()
axes[1].bar(avg_scores.index, avg_scores.values,
            color=['#d73027','#1a9641'], edgecolor='white', width=0.4)
axes[1].set_title('Avg Review Score\nLate vs On-Time', fontweight='bold')
axes[1].set_ylabel('Average Review Score')
axes[1].set_ylim(0, 5.5)
for i, v in enumerate(avg_scores.values):
    axes[1].text(i, v + 0.1, f'{v:.2f}', ha='center', fontsize=12, fontweight='bold')

# Chart 3: Review score breakdown by delivery type 
late_scores   = late_df[late_df['delivery_type']=='Late']['review_score'].value_counts().sort_index()
ontime_scores = late_df[late_df['delivery_type']=='On Time']['review_score'].value_counts().sort_index()

x = np.arange(5)
width = 0.35
axes[2].bar(x - width/2, (late_scores   / late_scores.sum() * 100).values,
            width, color='#d73027', edgecolor='white', label='Late', alpha=0.85)
axes[2].bar(x + width/2, (ontime_scores / ontime_scores.sum() * 100).values,
            width, color='#1a9641', edgecolor='white', label='On Time', alpha=0.85)
axes[2].set_title('Review Score % — Late vs On-Time', fontweight='bold')
axes[2].set_xlabel('Review Score')
axes[2].set_ylabel('% of Orders')
axes[2].set_xticks(x)
axes[2].set_xticklabels([1,2,3,4,5])
axes[2].legend()

plt.suptitle('Impact of Late Delivery on Customer Satisfaction', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('late_delivery_impact.png', dpi=150, bbox_inches='tight')
plt.show()




## Step 12 — Monthly Order Trend 

Usecase : Is the business growing over time?  
          This chart shows how total orders changed month by month.


In [ ]:
# Group by month and count orders
monthly = df.groupby('order_month').agg(
    total_orders    = ('order_id', 'count'),
    delivered_orders= ('order_status', lambda x: (x=='delivered').sum()),
    total_revenue   = ('total_price', 'sum')
).reset_index()

monthly['order_month_str'] = monthly['order_month'].astype(str)
monthly['delivery_rate']   = (monthly['delivered_orders'] / monthly['total_orders'] * 100).round(1)

# Show last 12 months for readability
monthly_recent = monthly.tail(20)

print(monthly_recent[['order_month_str','total_orders','delivered_orders','delivery_rate']].to_string(index=False))


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

x = range(len(monthly_recent))

# Chart 1: Monthly order volume 
axes[0].fill_between(x, monthly_recent['total_orders'], alpha=0.3, color='#2196F3')
axes[0].plot(x, monthly_recent['total_orders'], color='#2196F3',
             linewidth=2.5, marker='o', markersize=5)
axes[0].set_title('Monthly Order Volume', fontweight='bold')
axes[0].set_ylabel('Number of Orders')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(monthly_recent['order_month_str'], rotation=45, ha='right')

# Annotate peak
peak_idx = monthly_recent['total_orders'].idxmax()
peak_val = monthly_recent.loc[peak_idx, 'total_orders']
peak_x   = list(monthly_recent.index).index(peak_idx)
axes[0].annotate(f'Peak: {peak_val:,}',
                  xy=(peak_x, peak_val),
                  xytext=(peak_x - 2, peak_val + 200),
                  arrowprops=dict(arrowstyle='->', color='red'),
                  fontsize=10, color='red')

# Chart 2: Monthly delivery rate 
axes[1].bar(x, monthly_recent['delivery_rate'],
            color=['#1a9641' if r >= 90 else '#fdae61' if r >= 70 else '#d73027'
                   for r in monthly_recent['delivery_rate']],
            edgecolor='white')
axes[1].axhline(y=90, color='green',  linestyle='--', linewidth=1.5, label='90% target')
axes[1].axhline(y=70, color='orange', linestyle='--', linewidth=1.5, label='70% threshold')
axes[1].set_title('Monthly Delivery Rate (%)', fontweight='bold')
axes[1].set_ylabel('Delivery Rate (%)')
axes[1].set_ylim(0, 110)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(monthly_recent['order_month_str'], rotation=45, ha='right')
axes[1].legend()

plt.suptitle('Monthly Business Trend', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()




## Step 13 — Top Product Categories 

Usecase : Which product categories generate the most orders and revenue?  
          Which categories have the worst delivery rates?


In [ ]:
# Merge products into items
items_with_cat = items.merge(products[['product_id','product_category_name']],
                              on='product_id', how='left')
items_with_cat = items_with_cat.merge(orders[['order_id','order_status']],
                                       on='order_id', how='left')

# Top categories by order count
top_cat = (items_with_cat[items_with_cat['product_category_name'].notna()]
           .groupby('product_category_name')
           .agg(order_count = ('order_id', 'nunique'),
                total_revenue= ('price', 'sum'),
                delivery_rate= ('order_status', lambda x: (x=='delivered').mean()*100))
           .reset_index()
           .sort_values('order_count', ascending=False)
           .head(15))

print("=== Top 15 Categories by Orders ===")
print(top_cat[['product_category_name','order_count','total_revenue','delivery_rate']].to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
pal = sns.color_palette('tab20', 15)

# Chart 1: Top 15 categories by order count 
axes[0].barh(top_cat['product_category_name'][::-1],
             top_cat['order_count'][::-1], color=pal, edgecolor='white')
axes[0].set_title('Top 15 Product Categories by Orders', fontweight='bold')
axes[0].set_xlabel('Number of Orders')
for i, v in enumerate(top_cat['order_count'][::-1]):
    axes[0].text(v + 50, i, f'{v:,}', va='center', fontsize=8)

# Chart 2: Delivery rate per top category 
# Sort by delivery rate 
top_cat_sorted = top_cat.sort_values('delivery_rate')
bar_colors_dr  = ['#d73027' if v < 85 else '#1a9641' for v in top_cat_sorted['delivery_rate']]

axes[1].barh(top_cat_sorted['product_category_name'],
             top_cat_sorted['delivery_rate'],
             color=bar_colors_dr, edgecolor='white')
axes[1].axvline(x=90, color='green', linestyle='--', linewidth=2, label='90% target')
axes[1].set_title('Delivery Rate (%) by Category\n(Red = Below 85%)', fontweight='bold')
axes[1].set_xlabel('Delivery Rate (%)')
axes[1].set_xlim(0, 110)
axes[1].legend()
for i, v in enumerate(top_cat_sorted['delivery_rate']):
    axes[1].text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=8)

plt.suptitle('Product Category Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('category_analysis.png', dpi=150, bbox_inches='tight')
plt.show()




## Step 14 — Customer Repeat Behavior 

Usecase : How many customers bought more than once?  
          Repeat customers spend more and cost less to keep.  
          A low repeat rate means the business depends entirely on new customers.


In [ ]:
# Count orders per unique customer
cust_orders = (df[df['order_status'] == 'delivered']
               .merge(customers[['customer_id','customer_unique_id']], on='customer_id', how='left')
               .groupby('customer_unique_id')
               .agg(total_orders = ('order_id', 'count'),
                    total_spent   = ('total_price', 'sum'))
               .reset_index())

# Segment into buyer types
cust_orders['segment'] = pd.cut(
    cust_orders['total_orders'],
    bins=[0, 1, 2, 5, 9999],
    labels=['One-time Buyer', 'Bought Twice', '3-5 Orders', '6+ Orders']
)

seg_summary = cust_orders.groupby('segment').agg(
    customers   = ('customer_unique_id', 'count'),
    avg_spent   = ('total_spent', 'mean')
).reset_index()
seg_summary['pct'] = (seg_summary['customers'] / seg_summary['customers'].sum() * 100).round(1)

print("=== Customer Segment Summary ===")
print(seg_summary.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
seg_colors = ['#d73027','#fdae61','#4CAF50','#1a9641']

# Chart 1: Pie chart of customer segments 
axes[0].pie(seg_summary['customers'], labels=seg_summary['segment'],
            autopct='%1.1f%%', colors=seg_colors, startangle=140, pctdistance=0.75)
axes[0].set_title('Customer Segments\nby Purchase Frequency', fontweight='bold')

# Chart 2: Avg spend per segment 
axes[1].bar(seg_summary['segment'], seg_summary['avg_spent'],
            color=seg_colors, edgecolor='white')
axes[1].set_title('Average Total Spend by Segment', fontweight='bold')
axes[1].set_xlabel('Customer Segment')
axes[1].set_ylabel('Avg Total Spend (BRL)')
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(seg_summary['avg_spent']):
    axes[1].text(i, v + 5, f'R${v:.0f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Customer Repeat Behavior Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('repeat_customers.png', dpi=150, bbox_inches='tight')
plt.show()




## Step 15 — Key Findings Summary 

This is the final step — summarising everything we found.  



In [ ]:
print("=" * 60)
print("   E-COMMERCE FUNNEL EDA — KEY FINDINGS SUMMARY")
print("=" * 60)

# --- Funnel ---
s1 = len(df)
s4 = (df['order_status'] == 'delivered').sum()
s5 = df['review_score'].notna().sum()
print(f"\n📦 FUNNEL")
print(f"   Total Orders Placed  : {s1:,}")
print(f"   Orders Delivered     : {s4:,}  ({s4/s1*100:.1f}% conversion)")
print(f"   Reviews Left         : {s5:,}  ({s5/s4*100:.1f}% of delivered)")

# --- Revenue ---
total_rev = df[df['order_status']=='delivered']['total_price'].sum()
avg_order = df[df['order_status']=='delivered']['total_price'].mean()
print(f"\n REVENUE")
print(f"   Total Revenue        : R$ {total_rev:,.2f}")
print(f"   Avg Order Value      : R$ {avg_order:,.2f}")

# --- Delivery ---
delivered_df = df[(df['order_status']=='delivered') & df['days_to_deliver'].notna()]
avg_days = delivered_df['days_to_deliver'].mean()
late_pct = (delivered_df['is_late'] == 1).mean() * 100
print(f"\n DELIVERY SPEED")
print(f"   Avg Days to Deliver  : {avg_days:.1f} days")
print(f"   Late Delivery Rate   : {late_pct:.1f}%")

# --- Reviews ---
avg_rev  = df['review_score'].mean()
five_pct = (df['review_score'] == 5).sum() / df['review_score'].notna().sum() * 100
one_pct  = (df['review_score'] == 1).sum() / df['review_score'].notna().sum() * 100
print(f"\n REVIEWS")
print(f"   Avg Review Score     : {avg_rev:.2f} / 5.00")
print(f"   5-Star Reviews       : {five_pct:.1f}%")
print(f"   1-Star Reviews       : {one_pct:.1f}%")

# --- Repeat customers ---
repeat_pct = (cust_orders['total_orders'] > 1).mean() * 100
print(f"\n CUSTOMERS")
print(f"   Repeat Customer Rate : {repeat_pct:.1f}%")
print(f"   (Customers who bought more than once)")

print("\n" + "=" * 60)
print("   END OF EDA")
print("=" * 60)


---

## ✅ Project Complete!

### What you practised in this EDA:

| Skill | Where Used |
|---|---|
| `pd.read_csv()` | Loading data |
| `df.merge()` | Joining multiple tables |
| `pd.to_datetime()` | Converting date columns |
| `groupby().agg()` | Aggregating data |
| `pd.cut()` | Creating categories/buckets |
| `matplotlib` + `seaborn` | Visualisation |
| Funnel analysis logic | Steps 6, 7, 11 |
| Business thinking | Every step |

---

### 📂 Output files generated:
- `01_order_status.png`
- `02_funnel_analysis.png`
- `03_revenue_analysis.png`
- `04_delivery_time.png`
- `05_payment_analysis.png`
- `06_review_scores.png`
- `07_late_delivery_impact.png`
- `08_monthly_trend.png`
- `09_category_analysis.png`
- `10_repeat_customers.png`

---

> 🎯 **Pro tip for your portfolio:**  
> Screenshots of these charts + your SQL project together = a strong DA portfolio for a fresher.
